In [53]:
pip install google-cloud-bigquery pandas pyarrow db-dtypes requests

Note: you may need to restart the kernel to use updated packages.


In [54]:
import os

print("Project ID found:", bool(os.environ.get("gcl_project_id")))
print("GCP service account key found:", bool(os.environ.get("GCP_SA_KEY")))

Project ID found: True
GCP service account key found: True


In [55]:
import sys, json
import pandas as pd
from google.cloud import bigquery

PROJECT_ID = os.environ["gcl_project_id"]   # ← замінити на свій проєкт
LOCATION   = "EU"

creds = None

if "google.colab" in sys.modules:                  # Colab
    from google.colab import auth
    auth.authenticate_user()

elif os.environ.get("GCP_SA_KEY"):                 # Datalore: ключ у змінній середовища
    from google.oauth2 import service_account
    creds = service_account.Credentials.from_service_account_info(
        json.loads(os.environ["GCP_SA_KEY"]),
        scopes=["https://www.googleapis.com/auth/cloud-platform"])

client = bigquery.Client(project=PROJECT_ID, location=LOCATION, credentials=creds)
print("проєкт:", client.project)

проєкт: datalab-504011


In [56]:
# Завдання 4. Ноутбук 03 — вимір дат

# Завдання 4.1. Дізнайтеся мінімальну й максимальну business_date у nbu_raw.raw_rates одним запитом.

raw_rates = f"""
SELECT MIN(business_date) as min_date, MAX(business_date) as max_date
FROM `{PROJECT_ID}.nbu_raw.raw_rates`
""" 

df = client.query(raw_rates).to_dataframe()
df

/opt/python/envs/default_3_11/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,min_date,max_date
0,2026-08-24,2026-08-27


In [57]:
# Завдання 4.2. Побудуйте безперервний календар від 1 січня року мінімальної дати до 31 грудня

year_min = pd.to_datetime(df["min_date"].iloc[0]).year
year_max = pd.to_datetime(df["max_date"].iloc[0]).year

dates = pd.date_range(f"{year_min}-01-01", f"{year_max}-12-31", freq="D")


In [58]:
#  Завдання 4.3. Додайте колонки:
dim_dates = pd.DataFrame(dates, columns=["full_date"])

dim_dates["date_key"] = dim_dates["full_date"].dt.strftime("%Y%m%d").astype(int)

dim_dates["year"] = dim_dates["full_date"].dt.year
dim_dates["quarter"] = dim_dates["full_date"].dt.quarter
dim_dates["month"] = dim_dates["full_date"].dt.month
dim_dates["year_month"] = dim_dates["full_date"].dt.strftime("%Y-%m")
dim_dates["day_name"] = dim_dates["full_date"].dt.day_name()
dim_dates["is_weekend"] = dim_dates["full_date"].dt.dayofweek >= 5
dim_dates["full_date"] = dim_dates["full_date"].dt.date


dim_dates

,full_date,date_key,year,quarter,month,year_month,day_name,is_weekend
0,2026-01-01,20260101,2026,1,1,2026-01,Thursday,False
1,2026-01-02,20260102,2026,1,1,2026-01,Friday,False
2,2026-01-03,20260103,2026,1,1,2026-01,Saturday,True
3,2026-01-04,20260104,2026,1,1,2026-01,Sunday,True
4,2026-01-05,20260105,2026,1,1,2026-01,Monday,False
...,...,...,...,...,...,...,...,...
360,2026-12-27,20261227,2026,4,12,2026-12,Sunday,True
361,2026-12-28,20261228,2026,4,12,2026-12,Monday,False
362,2026-12-29,20261229,2026,4,12,2026-12,Tuesday,False
363,2026-12-30,20261230,2026,4,12,2026-12,Wednesday,False


In [59]:
# Завдання 4.4. Додайте рядок Unknown із date_key = -1, full_date = 1900-01-01 і текстовими полями Unknown.

unknown_raw = {
    "date_key": -1,
    "full_date": pd.to_datetime("1900-01-01").date(),
    "year": -1,
    "quarter": -1,
    "month": -1,
    "year_month": "Unknown",
    "day_name": "Unknown",
    "is_weekend": False
}

dim_dates.loc[len(dim_dates)] = unknown_raw

dim_dates = dim_dates.sort_values("date_key")
dim_dates

,full_date,date_key,year,quarter,month,year_month,day_name,is_weekend
365,1900-01-01,-1,-1,-1,-1,Unknown,Unknown,False
0,2026-01-01,20260101,2026,1,1,2026-01,Thursday,False
1,2026-01-02,20260102,2026,1,1,2026-01,Friday,False
2,2026-01-03,20260103,2026,1,1,2026-01,Saturday,True
3,2026-01-04,20260104,2026,1,1,2026-01,Sunday,True
...,...,...,...,...,...,...,...,...
360,2026-12-27,20261227,2026,4,12,2026-12,Sunday,True
361,2026-12-28,20261228,2026,4,12,2026-12,Monday,False
362,2026-12-29,20261229,2026,4,12,2026-12,Tuesday,False
363,2026-12-30,20261230,2026,4,12,2026-12,Wednesday,False


In [60]:
# Завдання 4.5. Запишіть у nbu_dwh.dim_date у режимі WRITE_TRUNCATE

dim_date_bq = f"{PROJECT_ID}.nbu_dwh.dim_date"


schema = [
    bigquery.SchemaField("date_key", "INT64"),
    bigquery.SchemaField("full_date", "DATE"),
    bigquery.SchemaField("year", "INT64"),
    bigquery.SchemaField("quarter", "INT64"),
    bigquery.SchemaField("month", "INT64"),
    bigquery.SchemaField("year_month", "STRING"),
    bigquery.SchemaField("day_name", "STRING"),
    bigquery.SchemaField("is_weekend", "BOOLEAN"),
]


cfg = bigquery.LoadJobConfig(schema=schema,
                            write_disposition="WRITE_TRUNCATE")

client.load_table_from_dataframe(dim_dates, dim_date_bq, job_config=cfg).result()



/opt/python/envs/default_3_11/lib/python3.11/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


LoadJob<project=datalab-504011, location=EU, id=656d8d59-d13a-427c-87f0-956722013dc6>

In [92]:
# Завдання 4.6. Перевірте й виведіть результат: 

query = f"""
SELECT *
FROM `{PROJECT_ID}.nbu_dwh.dim_date`
"""

dim_date_check = client.query(query).to_dataframe()

print("-1 in date_key: ", (dim_date_check["date_key"].isin([-1])).any())
print("How many times: ", (dim_date_check["date_key"] == -1).sum())

dates_check = dim_date_check[dim_date_check["date_key"] != -1].sort_values("full_date")

dates_check
print("More than 1 day difference:", (dates_check["full_date"].diff().dt.days > 1).any())


-1 in date_key:  True
How many times:  1
More than 1 day difference: False


/opt/python/envs/default_3_11/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


18 NaT
0 2026-01-01
9 2026-01-02
14 2026-01-03
5 2026-01-04
 ... 
349 2026-12-26
341 2026-12-27
359 2026-12-28
364 2026-12-29
354 2026-12-30
Name: full_date, Length: 365, dtype: dbdate